In [9]:
!pip install -q \
  torch \
  transformers \
  datasets \
  peft \
  bitsandbytes \
  accelerate \
  trl

In [10]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
from datasets import load_dataset
dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "val.jsonl"
    }
)


In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)
tokenizer.pad_token = tokenizer.eos_token


In [16]:
MAX_LENGTH = 256

def format_instruction(batch):
    prompts = []
    labels_list = []
    input_ids_list = []
    attention_masks = []

    for instruction, input_text, output in zip(
        batch["instruction"], batch["input"], batch["output"]
    ):
        if input_text.strip():
            prompt = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""
        else:
            prompt = f"""### Instruction:
{instruction}

### Response:
"""

        full_text = prompt + output

        tokenized = tokenizer(
            full_text,
            truncation=True,
            padding=False,
            max_length=MAX_LENGTH
        )

        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        response_start = len(prompt_ids)

        labels = [-100] * response_start + tokenized["input_ids"][response_start:]
        labels = labels[:MAX_LENGTH]

        input_ids_list.append(tokenized["input_ids"])
        attention_masks.append(tokenized["attention_mask"])
        labels_list.append(labels)

    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_masks,
        "labels": labels_list
    }




In [17]:
tokenized_ds = dataset.map(
    format_instruction,
    remove_columns=dataset["train"].column_names,
    batched=True
)


Map:   0%|          | 0/929 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

In [18]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [19]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


**LORA configurations:-**

In [20]:
model.gradient_checkpointing_enable()


In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=500,
    eval_strategy="steps",
    save_strategy="epoch",
    eval_steps=500,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)


**TRAINER Initialization:-**

In [25]:
from transformers import Trainer, DataCollatorForSeq2Seq


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding=True,
        pad_to_multiple_of=8
    )
)
trainer.train()


Step,Training Loss,Validation Loss
500,0.244274,0.491329


TrainOutput(global_step=699, training_loss=0.44706862099010375, metrics={'train_runtime': 267.3516, 'train_samples_per_second': 10.424, 'train_steps_per_second': 2.615, 'total_flos': 1265915172028416.0, 'train_loss': 0.44706862099010375, 'epoch': 3.0})

**Train model:-**

In [26]:
adapter_path = "./adapters"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

('./adapters/tokenizer_config.json',
 './adapters/chat_template.jinja',
 './adapters/tokenizer.json')

In [27]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config
)

model = PeftModel.from_pretrained(base_model, adapter_path)
prompt = """### Instruction:
Answer the medical question accurately.

### Input:
Explain congestive heart failure.
### Response:
"""
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = decoded.split("### Response:")[-1].strip()

print("\n" + "=" * 80)
print("MODEL RESPONSE")
print("=" * 80)
print(response)
print("=" * 80)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


MODEL RESPONSE
congestive heart failure is a chronic condition where the heart can't pump enough blood, causing fatigue and shortness of breath. It is caused by reduced heart muscle function or weakened valves.


In [28]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "./adapters"
MERGED_PATH = "./quantized/fp16-merged"

In [30]:
from peft import PeftModel
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model = model.merge_and_unload()

model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./quantized/fp16-merged/tokenizer_config.json',
 './quantized/fp16-merged/chat_template.jinja',
 './quantized/fp16-merged/tokenizer.json')

In [31]:
from transformers import BitsAndBytesConfig

int8_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_int8 = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    device_map="auto",
    quantization_config=int8_config
)

model_int8.save_pretrained("./quantized/model-int8")
tokenizer.save_pretrained("./quantized/model-int8")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./quantized/model-int8/tokenizer_config.json',
 './quantized/model-int8/chat_template.jinja',
 './quantized/model-int8/tokenizer.json')

In [32]:
int4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    device_map="auto",
    quantization_config=int4_config
)

model_int4.save_pretrained("./quantized/model-int4")
tokenizer.save_pretrained("./quantized/model-int4")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./quantized/model-int4/tokenizer_config.json',
 './quantized/model-int4/chat_template.jinja',
 './quantized/model-int4/tokenizer.json')

**DAY-3:-**

In [33]:
!git clone https://github.com/ggerganov/llama.cpp.git


fatal: destination path 'llama.cpp' already exists and is not an empty directory.


In [34]:
!python llama.cpp/convert_hf_to_gguf.py \
  ./quantized/fp16-merged \
  --outfile ./quantized/model-fp16.gguf

INFO:hf-to-gguf:Loading model: fp16-merged
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:heuristics detected float16 tensor dtype, setting --outtype f16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         

In [35]:
!cd llama.cpp && mkdir -p build && cd build && cmake .. && make -j1

CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.9.5
-- ggml commit:  1e8924fd6
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: common
-- Configuring done (0.7s)
-- Generating done (0.5s)
-- Build files have been written to: /content/llama.cpp/build
[  2%] Built target ggml-base
[  6%] Built target ggml-cpu
[  7%] Built target ggml
[ 39%] Built target llama
[ 39%] Built target build_info
[ 39%] Built target cpp-httplib
[ 40%] Building CXX object common/CMakeFiles/common.dir/__/license.cpp.o
[ 40%] Linking CXX static library libcommon.a
[ 46%] Built target common
[ 46%] Building CXX object tests/CMakeFiles/test-tokenizer-0.dir/test-tokenizer-0.cpp.o
[ 46%] Linking CXX executable ../bin/test-token

In [36]:
! llama.cpp/build/bin/llama-quantize \
  ./quantized/model-fp16.gguf \
  ./quantized/model.gguf \
  q4_0

main: build = 7974 (1e8924fd6)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing './quantized/model-fp16.gguf' to './quantized/model.gguf' as Q4_0
llama_model_loader: loaded meta data with 31 key-value pairs and 201 tensors from ./quantized/model-fp16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Fp16 Merged
llama_model_loader: - kv   3:                           general.finetune str              = merged
llama_model_loader: - kv   4:                         general.size_label str              = 1.1B
llama_model_loader: - kv   5:                          llama.block_count u32              = 22
llama_model_loa

In [37]:
! llama.cpp/build/bin/llama-cli \
  -m ./quantized/model.gguf \
  -p "Explain gradient checkpointing in simple terms." \
  -n 128



Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\| 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b7974-1e8924fd6
model      : model.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> Explain gradient checkpointing in simple terms.

|-\|/-\|/-\|/-\ Gradient checkpointing is a process in which the code and data of a running program are saved periodically to a backup store or archive, such as a file system, database, or cloud storage, to ensure the program's smooth execution in case of unexpected system failures. It helps prevent data loss or corruption if 

In [38]:
from datasets import load_dataset

val_ds = load_dataset(
    "json",
    data_files="val.jsonl"
)["train"]

sample = val_ds[1]

instruction = sample["instruction"]
input_text = sample["input"]


Generating train split: 0 examples [00:00, ? examples/s]

In [39]:
def build_prompt(instruction, input_text):
    if input_text.strip():
        return f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""
    else:
        return f"""### Instruction:
{instruction}

### Response:
"""


In [40]:

prompt = build_prompt(instruction, input_text)
print(prompt)


### Instruction:
A patient presents with fever, chills, and right flank pain. Urinalysis shows bacteria and white blood cells. What is the diagnosis?

### Response:



In [41]:
import time
import torch

def measure_speed(model, tokenizer, prompt, max_new_tokens=100, runs=3):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # warmup
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=20)

    times = []
    for _ in range(runs):
        start = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=max_new_tokens)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    return max_new_tokens / avg_time  # tokens/sec


In [42]:
# FP16 merged model
model_fp16 = AutoModelForCausalLM.from_pretrained(
    "./quantized/fp16-merged",
    dtype=torch.float16,
    device_map="auto"
)

# INT8 model
model_int8 = AutoModelForCausalLM.from_pretrained(
    "./quantized/model-int8",
    device_map="auto"
)

# INT4 model
model_int4 = AutoModelForCausalLM.from_pretrained(
    "./quantized/model-int4",
    device_map="auto"
)



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [43]:
speed_fp16 = measure_speed(model_fp16, tokenizer, prompt)
speed_int8 = measure_speed(model_int8, tokenizer, prompt)
speed_int4 = measure_speed(model_int4, tokenizer, prompt)


print("FP16:", speed_fp16)
print("INT8:", speed_int8)
print("INT4:", speed_int4)



FP16: 33.91803331715995
INT8: 9.605678821153049
INT4: 17.09004494320736


In [44]:
! llama.cpp/./build/bin/llama-cli \
  -m quantized/model.gguf \
  -p f"{prompt}"\
  -n 100


Loading model... |-\|/-\|/-\ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b7974-1e8924fd6
model      : model.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> f### Instruction:
A patient presents with fever, chills, and right flank pain. Urinalysis shows bacteria and white blood cells. What is the diagnosis?

### Response:


|-\|/-\|/-\|/-\|/-\|/-\| This patient has pyelonephritis with bacterial infection and white blood cells. The disease is caused by Chlamydia or E. Coli. Treatment includes antibiotics for bacterial infection.

[ Prompt: 27.8 t/s | Generation: 12.5 t/s

In [45]:
!pip install torch transformers peft accelerate psutil llama-cpp-python sentence-transformers pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 21.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.3 MB/s eta 0:00:00


In [46]:
GGUF_MODEL = "./quantized/model.gguf"
PROMPTS = [
    """### Instruction:
Answer the medical question accurately.

### Input:
What is the treatment for chronic back pain?

### Response:
""",

    """### Instruction:
Answer the medical question accurately.

### Input:
What is COPD?

### Response:
""",

    """### Instruction:
Answer the medical question accurately.

### Input:
What are the symptoms of anemia?

### Response:
"""
]


GROUND_TRUTH = [
    "Treatment includes physical therapy, pain medications, exercise, lifestyle modifications, and sometimes surgical interventions.",

    "Chronic Obstructive Pulmonary Disease (COPD) is a group of progressive lung diseases, including emphysema and chronic bronchitis, causing breathing difficulties.",

    "Common symptoms of anemia include fatigue, weakness, pale skin, shortness of breath, dizziness, and a rapid heartbeat."
]


In [47]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [49]:
def get_vram():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2
    return 0

def accuracy(preds, refs):
    p_emb = embedder.encode(preds, convert_to_tensor=True)
    r_emb = embedder.encode(refs, convert_to_tensor=True)

    sims = util.cos_sim(p_emb, r_emb)

    per_sample_scores = sims.diag().cpu().numpy()

    # for i, score in enumerate(per_sample_scores):
    #     print(f"Sample {i+1} Similarity: {score:.3f}")

    mean_score = per_sample_scores.mean()

    return mean_score

In [55]:
from sentence_transformers import util

from llama_cpp import Llama

def benchmark_gguf(label):
    llm = Llama(model_path=GGUF_MODEL, n_ctx=2048, n_threads=8, verbose=False)

    outputs = []
    start = time.time()

    for p in PROMPTS:
        response = ""
        stream = llm(p, max_tokens=256, stream=True)

        for output in stream:
            token = output["choices"][0]["text"]
            response += token
            # print(token, end="", flush=True)

        # print("\n \n")
        outputs.append(response)

    end = time.time()

    tokens = sum(len(o.split()) for o in outputs)
    tps = tokens / (end - start)
    acc = accuracy(outputs, GROUND_TRUTH)

    return {
        "Model": label,
        "Tokens/sec": round(tps, 2),
        "Latency(s)": round(end - start, 2),
        "VRAM(MB)": 0,
        "Accuracy": round(acc, 3)
    }

results = []

results.append(benchmark_gguf("GGUF Q4 llama.cpp"))

In [56]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
import pandas as pd
import threading

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
FT_MODEL = "./quantized/fp16-merged"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_PATH = "results.csv"



def benchmark_hf(model_path, label):

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, device_map=DEVICE)

    outputs = []
    start = time.time()

    for prompt in PROMPTS:
        streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

        generation_kwargs = dict(
            **inputs,
            max_new_tokens=256,
            streamer=streamer,
            pad_token_id=tokenizer.eos_token_id
        )

        thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()

        response = ""
        for token in streamer:
            response += token
        #     print(token, end="", flush=True)

        # print("\n \n")
        outputs.append(response)
        thread.join()

    end = time.time()

    total_tokens = sum(len(tokenizer.encode(r)) for r in outputs)
    duration = end - start
    tps = total_tokens / duration
    acc = accuracy(outputs, GROUND_TRUTH)

    return {
        "Model": label,
        "Tokens/sec": round(tps, 2),
        "Latency(s)": round(duration, 2),
        "VRAM(MB)": round(get_vram(), 2),
        "Accuracy": round(acc, 3)
    }


results.append(benchmark_hf(BASE_MODEL, "Base Model"))
results.append(benchmark_hf(FT_MODEL, "Fine-tuned"))

df = pd.DataFrame(results)
df.to_csv(RESULTS_PATH, index=False)

print(df)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

               Model  Tokens/sec  Latency(s)  VRAM(MB)  Accuracy
0  GGUF Q4 llama.cpp        4.42       37.74      0.00     0.807
1         Base Model       43.26        8.14  11186.47     0.785
2         Fine-tuned       36.42       24.00  11186.47     0.680
